# Main Figure 1

Show one example of a network and its biomass-richness relationship.

Input: generated/simulations.csv and generated/run_config.json. Run 00_generate_simulations.ipynb first. Set DATA_DIR for a different run.

The plot uses one degree and one interaction strength.

## Settings

Change DATA_DIR and SAVE in the next cell.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

DATA_DIR = Path('generated')
SAVE = True
FIGURE_DIR = Path('figures')
FOCUS_DEGREE = 1.0
FOCUS_STRENGTH = 0.5
simulations = pd.read_csv(DATA_DIR / 'simulations.csv')
config = json.loads((DATA_DIR / 'run_config.json').read_text(encoding='utf-8'))
FUNCTION_COL = 'biomass_for_bef'
if config['profile'] != 'final_10000':
    raise ValueError('Use data from the final_10000 simulation notebook')

def slope_table(df, x='N', y=None, groups=('degree','strength')):
    y = FUNCTION_COL if y is None else y
    rows = []
    for keys, part in df.groupby(list(groups)):
        if part[x].nunique() < 2:
            continue
        fit = linregress(part[x], part[y])
        rows.append(dict(zip(groups, keys if isinstance(keys, tuple) else (keys,))) |
                    dict(slope=fit.slope, intercept=fit.intercept, stderr=fit.stderr,
                         n=len(part), richness_levels=part[x].nunique()))
    return pd.DataFrame(rows)

def set_strength_axis(ax, values):
    lo, hi = float(np.min(values)), float(np.max(values))
    ticks = sorted(set([lo, hi] + [float(x) for x in np.linspace(.2, 1, 5) if lo < x < hi]))
    ax.set_xlim(lo, hi)
    ax.set_xticks(ticks, [f'{x:g}' for x in ticks])

def finish(fig, stem):
    fig.tight_layout()
    if SAVE:
        FIGURE_DIR.mkdir(exist_ok=True)
        fig.savefig(FIGURE_DIR / f'{stem}.png', dpi=300, bbox_inches='tight')
        fig.savefig(FIGURE_DIR / f'{stem}.pdf', bbox_inches='tight')
    plt.show()


## Figure

In [ ]:
chosen_strength = min(simulations.strength.unique(), key=lambda value: abs(value - FOCUS_STRENGTH))
chosen = simulations[simulations.strength == chosen_strength]
print('Displayed interaction strength:', chosen_strength)
fig, ax = plt.subplots(figsize=(6, 4))
for degree, part in chosen.groupby('degree'):
    means = part.groupby('N')[FUNCTION_COL].mean()
    ax.plot(means.index, means.values, 'o-', label=f'k={degree:g}')
ax.set(xlabel='Initial richness', ylabel='Total biomass used for BEF')
ax.legend(title='Mean degree')
print('Displayed runs:', len(chosen))
finish(fig, 'main_figure_1')